In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV
import joblib
import re
import string
import nltk
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_true=pd.read_csv('DataSet\True.csv')
df_fake=pd.read_csv('DataSet\Fake.csv')

In [3]:
df_true["class"]=1
df_fake["class"]=0

In [4]:
df=pd.concat([df_true,df_fake],ignore_index=True)

In [5]:
df.head()

,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   class    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [7]:
df["subject"].value_counts()

subject
politicsNews       11272
worldnews          10145
News                9050
politics            6841
left-news           4459
Government News     1570
US_News              783
Middle-east          778
Name: count, dtype: int64

In [8]:
df.drop(["date","title","subject"],axis=1,inplace=True)

In [9]:
df.head()

,text,class
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


In [10]:
df

,text,class
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1
...,...,...
44893,21st Century Wire says As 21WIRE reported earl...,0
44894,21st Century Wire says It s a familiar theme. ...,0
44895,Patrick Henningsen 21st Century WireRemember ...,0
44896,21st Century Wire says Al Jazeera America will...,0


In [11]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\razas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\razas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\razas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\razas\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
def clean_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]','',text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+','',text)
    text = re.sub('<.*?>+',b'',text)
    text = re.sub('[%s]' % re.escape(string.punctuation),'',text)
    text = re.sub('\w*\d\w*','',text)
    return text

In [13]:
df["text"]=df["text"].apply(clean_text)

In [14]:
# remove stopword
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
df['text']=df['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word.lower() not in stop_words]))

In [15]:
df["text"]

0        washington reuters head conservative republica...
1        washington reuters transgender people allowed ...
2        washington reuters special counsel investigati...
3        washington reuters trump campaign adviser geor...
4        seattle washington reuters president donald tr...
                               ...                        
44893    century wire says reported earlier week unlike...
44894    century wire says familiar theme whenever disp...
44895    patrick henningsen century wireremember obama ...
44896    century wire says al jazeera america go histor...
44897    century wire says predicted new year look ahea...
Name: text, Length: 44898, dtype: object

In [16]:
X=df["text"]
y=df["class"]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
vectorizer=TfidfVectorizer()

In [19]:
vectorizer.fit(X_train)
X_train_vect = vectorizer.transform(X_train)
X_test_vect = vectorizer.transform(X_test)
joblib.dump(vectorizer,"vectorizer.pkl")

['vectorizer.pkl']

In [29]:
def create_model(X_train_vect,y_train,X_test_vect,y_test):
    models = {
        "Logistic Regression": LogisticRegression(),
        "Support Vector Machine": SVC(),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Naive Bayes": MultinomialNB(),
        "Gradient Boosting": GradientBoostingClassifier(),
        "XGBoost": XGBClassifier()
    }
    for name, model in models.items():
        model.fit(X_train_vect, y_train)
        print(f"{name} model created and trained.")
    print("Result")
    for name, model in models.items():
        y_pred = model.predict(X_test_vect)
        print(f"{name} Accuracy: {accuracy_score(y_test, y_pred)}")
        print(classification_report(y_test, y_pred))
        print(confusion_matrix(y_test, y_pred))



In [ ]:
create_model(X_train_vect,y_train,X_test_vect,y_test)

Logistic Regression model created and trained.


In [30]:
lr=LogisticRegression()
lr.fit(X_train_vect,y_train)

LogisticRegression()

In [31]:
y_pred_lr=lr.predict(X_test_vect)

In [32]:
print(accuracy_score(y_test,y_pred_lr))
print(classification_report(y_test,y_pred_lr))
print(confusion_matrix(y_test,y_pred_lr))

0.9889755011135858
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4650
           1       0.99      0.99      0.99      4330

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

[[4589   61]
 [  38 4292]]


In [33]:
svc=SVC()
svc.fit(X_train_vect,y_train)

SVC()

In [34]:
y_pred_svc=svc.predict(X_test_vect)
print(accuracy_score(y_test,y_pred_svc))
print(classification_report(y_test,y_pred_svc))
print(confusion_matrix(y_test,y_pred_svc))


0.994543429844098
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      4650
           1       0.99      1.00      0.99      4330

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

[[4619   31]
 [  18 4312]]


In [35]:
dt=DecisionTreeClassifier()
dt.fit(X_train_vect,y_train)

DecisionTreeClassifier()

In [36]:
y_pred_dt=dt.predict(X_test_vect)
print(accuracy_score(y_test,y_pred_dt))
print(classification_report(y_test,y_pred_dt))
print(confusion_matrix(y_test,y_pred_dt))


0.9957683741648107
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4650
           1       1.00      1.00      1.00      4330

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

[[4632   18]
 [  20 4310]]


In [37]:
rf=RandomForestClassifier()
rf.fit(X_train_vect,y_train)

RandomForestClassifier()

In [38]:
y_pred_rf=rf.predict(X_test_vect)
print(accuracy_score(y_test,y_pred_rf))
print(classification_report(y_test,y_pred_rf))
print(confusion_matrix(y_test,y_pred_rf))

0.993652561247216
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4650
           1       0.99      0.99      0.99      4330

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

[[4620   30]
 [  27 4303]]


In [39]:
nv=MultinomialNB()
nv.fit(X_train_vect,y_train)

MultinomialNB()

In [40]:
y_pred_nv=nv.predict(X_test_vect)
print(accuracy_score(y_test,y_pred_nv))
print(classification_report(y_test,y_pred_nv))
print(confusion_matrix(y_test,y_pred_nv))


0.9365256124721604
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      4650
           1       0.94      0.93      0.93      4330

    accuracy                           0.94      8980
   macro avg       0.94      0.94      0.94      8980
weighted avg       0.94      0.94      0.94      8980

[[4378  272]
 [ 298 4032]]


In [41]:
xgb=XGBClassifier()
xgb.fit(X_train_vect,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [42]:
y_pred_xgb=xgb.predict(X_test_vect)
print(accuracy_score(y_test,y_pred_xgb))
print(classification_report(y_test,y_pred_xgb))
print(confusion_matrix(y_test,y_pred_xgb))

0.9971046770601336
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4650
           1       1.00      1.00      1.00      4330

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

[[4635   15]
 [  11 4319]]


In [43]:
Gbc=GradientBoostingClassifier()
Gbc.fit(X_train_vect,y_train)

GradientBoostingClassifier()

In [44]:
y_pred_Gbc=Gbc.predict(X_test_vect)
print(accuracy_score(y_test,y_pred_Gbc))
print(classification_report(y_test,y_pred_Gbc))
print(confusion_matrix(y_test,y_pred_Gbc))

0.9953229398663697
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      4650
           1       0.99      1.00      1.00      4330

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

[[4620   30]
 [  12 4318]]


In [46]:
joblib.dump(xgb,"final_model(XGBoost).pkl")

['final_model(XGBoost).pkl']